<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week01_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> <파이썬 머신러닝 완벽 가이드> 8장 p.488~508

# 8. 텍스트 분석

NLP: 언어 해석 중점, 기계번역, 질의응답 시스템 등  
텍스트 분석: 비정형 텍스트에서 의미 있는 정보 추출에 중점  

-> 머신러닝, 언어 이해, 통계 등을 활용해 모델을 수립하고 정보를 추출해 비즈니스 인텔리전스(Business Intelligence)예측 분석 등의 분석 작업을 주로 수행

- 텍스트 분류
- 감성 분석
- 텍스트 요약
- 텍스트 군집화, 유사도 측정

## 8-01 텍스트 분석 이해

비정형 데이터 처리(피처 형태 추출하는 방법과 의미있는 값 도출)가 중요  
**'피처 벡터화'('피처 추출')**  
- BOW(Bag of Words)
- Word2Vec  

### 텍스트 분석 수행 프로세스  
1. 텍스트 전처리 = 텍스트 정규화
- 클렌징(대/소문자 변경/ 특수문자 삭제)
- 토큰화 작업
- 무의미한 단어 제거
- 어근 추출
2. 피처 벡터화/추출  
: 피처 추출, 벡터값 할당
- BOW(Bag of Words)
- Word2Vec
3. ML 모델 수립 및 학습/예측/평가  
: ML모델 적용해서 수행

### 파이썬 기반의 NLP, 텍스트 분석 패키지

- NLTK : 가장 대표적, 수행속도 아쉬움
- Genism : 토픽 모델링 분야
- SpaCy : 최근 주목 받고 있음  


## 8-02 텍스트 사전 준비 작업(텍스트 전처리) - 텍스트 정규화

- 클렌징(대/소문자 변경/ 특수문자 삭제)
- 토큰화
- 무의미한 단어 제거/필터링/철자 수정
- 어근 추출
- Lemmatization

### 클렌징

불필요한 문자, 기호 삭제

### 텍스트 토큰화
- 문장 토큰화: 문서에서 문장 분리  
문장 마침표(.), 개행문자(\n) 등 기호에 따라 분리  
각 문장이 가지는 시맨틱적(맥락) 의미가 중요할 때 사용  













In [2]:
from nltk import sent_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

text_sample = 'The Matrix is everywhere its all around us, here even in this room. \
You can see it out your window or on your television. \
You feel it when you go to work, or go to church or pay your taxes.'
sentences = sent_tokenize(text=text_sample)
print(type(sentences), len(sentences))
print(sentences)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


<class 'list'> 3
['The Matrix is everywhere its all around us, here even in this room.', 'You can see it out your window or on your television.', 'You feel it when you go to work, or go to church or pay your taxes.']


`sent_tokenize()`: 각각의 문장으로 구성된 list객체 반환  
- 단어 토큰화: 문장을 단어로 분리  
단어 순서 중요하지 않은 경우 이거만 사용해도 충분!

`word_tokenize()`


In [3]:
from nltk import word_tokenize

sentence = 'The Matrix is everywhere its all around us, here even in this room.'
words = word_tokenize(sentence)
print(type(words), len(words))
print(words)

<class 'list'> 15
['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.']


둘 다 조합해 모든 단어 토큰화!  
문장별 단어 토큰화 적용: sent_ -> word_   ??  

문서 문장으로 나눈 뒤, 개별 문장을 다시 단어로 토큰화  



In [4]:
# 여러 개의 문장으로 된 입력 데이터를 문장별로 단어 토큰화하게 만드는 함수 생성
def tokenize_text(text):

  # 문장별로 분리 토큰
  sentences = sent_tokenize(text)
  # 분리된 문장별 단어 토큰화
  word_tokens = [word_tokenize(sentence) for sentence in sentences]
  return word_tokens

# 여러 문장에 대해 문장별 단어 토큰화 수행.
word_tokens = tokenize_text(text_sample)
print(type(word_tokens), len(word_tokens))
print(word_tokens)

<class 'list'> 3
[['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.'], ['You', 'can', 'see', 'it', 'out', 'your', 'window', 'or', 'on', 'your', 'television', '.'], ['You', 'feel', 'it', 'when', 'you', 'go', 'to', 'work', ',', 'or', 'go', 'to', 'church', 'or', 'pay', 'your', 'taxes', '.']]


단어별 토큰화하면 문맥 의미 무시되는 문제 발생  

`n-gram` : 연속된 n개의 단어를 하나의 토큰화 단위로 분리해 내는 것

### 스톱 워드 제거
Stop sord : 분석에 큰 (문맥적) 의미가 없는 단어  
예) is, the, a, will ...

필수 문법 요소인 경우가 많아 빈도 높음, 오인 방지 위해 사전에 제거   

언어별로 스톱어드 목록화되어있음!



In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
print('영어 stop words 개수:', len(nltk.corpus.stopwords.words('english')))
print(nltk.corpus.stopwords.words('english')[:20])

영어 stop words 개수: 198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


`corpus`: '말뭉치' 라는 뜻, 텍스트 데이터 집합을 의미함. NLTK 라이브러리에서 corpus 형태로 저장해두고 있음

위 예제에서 토큰화 한 리스트에서 스톱워드 필터링으로 제거해보기

In [7]:
stopwords = nltk.corpus.stopwords.words('english')
all_tokens = []
# 위 예제에서 3개의 문장별로 얻은 word_tokens list에 대해 스톱워드를 제거하는 반복문
for sentence in word_tokens:
  filtered_words=[]
  # 개별 문장별로 토큰화된 문장 list에 대해 스톱 워드를 제거하는 반복문
  for word in sentence:
    # 소문자로 모두 변환
    word = word.lower()
    # 토큰화된 개별 단어가 스톱 워드의 단어에 포함되지 않으면 word_tokens에 추가
    if word not in stopwords:
      filtered_words.append(word)
  all_tokens.append(filtered_words)

print(all_tokens)

[['matrix', 'everywhere', 'around', 'us', ',', 'even', 'room', '.'], ['see', 'window', 'television', '.'], ['feel', 'go', 'work', ',', 'go', 'church', 'pay', 'taxes', '.']]


`append`: list의 맨 마지막에 새로운 요소를 추가  
`lower`: 대문자를 소문자로 변경

### Stemming과 Lemmatization

문법적 또는 의미적으로 변화하는 단어의 원형을 찾는 것  
예) 동사원형, 과거형, 복수형, 진행형 등 ...


Stemming: 일반적 방법, 단순화된 방법 적용, 일부 철자 훼손된 어근 단어 추출하는 경향 있음
- Porter, Lancaster, Snowball Stemmer

Lemmatization: 더 정교함, 의미론적 기반으로 단어 원형 찾음. 품사/의미적 부분 감안, 정확하게 찾음, 시간 더 걸림.  
- WordNetLemmatizer  

stemmer 객체 생성 뒤 `객체.stem('단어')`




In [8]:
from nltk.stem import LancasterStemmer
stemmer = LancasterStemmer()

print(stemmer.stem('working'), stemmer.stem('works'), stemmer.stem('worked'))
print(stemmer.stem('amusing'), stemmer.stem('amuses'), stemmer.stem('amused'))
print(stemmer.stem('happier'), stemmer.stem('happiest'))
print(stemmer.stem('fancier'), stemmer.stem('fanciest'))

work work work
amus amus amus
happy happiest
fant fanciest


stem: work 말고도 원형 제대로 파악 못 함...  

In [9]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

lemma = WordNetLemmatizer()
print(lemma.lemmatize('amusing', 'v'), lemma.lemmatize('amuses','v'), lemma.lemmatize('amused', 'v'))
print(lemma.lemmatize('happier', 'a'), lemma.lemmatize('happiest', 'a'))
print(lemma.lemmatize('fancier','a'), lemma.lemmatize('fanciest', 'a'))

[nltk_data] Downloading package wordnet to /root/nltk_data...


amuse amuse amuse
happy happy
fancy fancy


WordNetLemmatizer이 확실히 정확함  
단, 단어 품사도 입력해줘야됨


## 8-03 Bag of Words — BOW

문서가 가지는 모든 단어(Words)를 문맥이나 순서를 무시하고 일괄적으로 단어에 대해 빈도 값을 부여해 피처 값을 추출

1. 모든 단어에서 중복 제거, 각 단어의 인덱스 부여
2. 개별 문장에서 단어의 빈도수를 인덱스에 기재함.

- 장점: 쉽고 빠른 구축.
- 단점: 문맥 의미 반영 부족 / 희소 행렬 문제(대부분이 0으로 채워짐)  

### BOW 피처 벡터화  

텍스트 (비정형 데이터) -> 꼭 숫자형 값, 벡터 값으로 변환 필요!!  
'피처 벡터화'(피처 추출에 포함됨)  

M개의 문서에서 N개의 단어 나오면  
M X N 크기의 행렬 추출됨.

- 카운트 기반의 벡터화: 카운트 값 높을수록 중요, 자주 사용되는 값일 뿐일 가능성 있음
- TF-IDF(Term Frequency — Inverse Document Frequency) 기반의 벡터화: 개별 문서 고빈도 단어에 높은 가중치, 모든 문서 전반적 빈도에 페널티, 더 높은 예측 성능 보장

### 사이킷런의 Count 및 TF-IDF 벡터화 구현: CountVectorizer, TfidfVectorizer

CountVectorizer 클래스: 카운트 기반 벡터화  
텍스트 전처리도 함께 수행  

1. 전처리(소문자 변경 등)
2. 토큰화 : n_gram_range 반영
3. 텍스트 정규화: Stemming과 Lemmatization 지원X, 함수 만들거나 외부패키지 이용해야함
4. 피처 벡터화

TfidfVectorizer 클래스: TF-IDF 기반 벡터화

### BOW 벡터화를 위한 희소 행렬

벡터화 후 CSR 형태의 희소 행렬을 반환.  
형태를 알아야 고난도 ML 모델 수립 가능  
희소행렬 단점(메모리 공간 차지, 연산 시간 소요) 해결 위해 물리적으로 적은 메모리 공간을 차지할 수 있도록 변환!  
- COO 형식
- CSR 형식: 저장, 계산 수행 능력 더 뛰어남

### 희소 행렬 —C00 형식
COO(Coordinate: 좌표) 형식  
0이 아닌 데이터만 별도의 데이터 배열(Array)에 저장, 그 데이터가 가리키는 행과 열의 위치를 별도의 배열로 저장하는 방식

파이썬 - 사이파이(Scipy)를 이용.  
sparse 패키지  

`sparse.coo_matrix((데이터, (행 위치, 열 위치)))`


In [11]:
import numpy as np

dense = np.array([[3,0,1], [0,2,0]])

In [12]:
from scipy import sparse

# 0이 아닌 데이터 추출
data = np.array([3,1,2])

# 행 위치와 열 위치를 각각 배열로 생성
row_pos = np.array([0,0,1])
col_pos = np.array([0,2,1])

# sparse 패키지의 coo_matrix를 이용해 COO 형식으로 희소 행렬 생성
sparse_coo = sparse.coo_matrix((data, (row_pos, col_pos)))

sparse_coo는 COO 형식의 희소 행렬 객체 변수.  
`toarray()`: 다시 밀집 형태 행렬로 출력

In [13]:
sparse_coo.toarray()

array([[3, 0, 1],
       [0, 2, 0]])

### 희소 행렬 - CSR 형식
CSR(Compressed Sparse Row)  
행 위치 배열 내에 있는 고유한 값의 시작 위치만 다시 별도의 위치 배열로 가지는 변환 방식

COO 형식이 행과 열의 위치를 나타내기 위해서 반복적인
위치 데이터를 사용해야 하는 문제점을 해결  


In [14]:
[[0,0,1,0,0,5], [1,4,0,3,2,5], [0,6,0,3,0,0], [2,0,0,0,0,0], [0,0,0,7,0,8], [1,0,0,0,0,0]]

[[0, 0, 1, 0, 0, 5],
 [1, 4, 0, 3, 2, 5],
 [0, 6, 0, 3, 0, 0],
 [2, 0, 0, 0, 0, 0],
 [0, 0, 0, 7, 0, 8],
 [1, 0, 0, 0, 0, 0]]

여기서 행, 열 위치 배열 확인하면 행 값 반복적으로 사용되는 구간 있음.  
행 위치 배열이 0부터 순차적으로 증가하는 값으로 이뤄졌다는 특성을 고려해 해결.  

[행 위치 배열의 고유값의 시작 인덱스 표기 + 총 항목 개수 배열]

`sparse.csr_matrix((데이터, 열 위치, 행 시작 위치 인덱스))`


In [16]:
from scipy import sparse

dense2 = np.array([[0,0,1,0,0,5], [1,4,0,3,2,5], [0,6,0,3,0,0], [2,0,0,0,0,0], [0,0,0,7,0,8], [1,0,0,0,0,0]])

# 0이 아닌 데이터 추출
data2 = np.array([1,5,1,4,3,2,5,6,3,2,7,8,1])

# 행 위치와 열 위치를 각각 array로 생성
row_pos = np.array([0,0,1,1,1,1,1,2,2,3,4,4,5])
col_pos = np.array([2,5,0,1,3,4,5,1,3,0,3,5,0])

# COO 형식으로 변환
sparse_coo = sparse.coo_matrix((data2, (row_pos, col_pos)))

# 행 위치 배열의 고유한 값의 시작 위치 인덱스를 배열로 생성
row_pos_ind = np.array([0,2,7,9,10,12,13])

# CSR 형식으로 변환
sparse_csr = sparse.csr_matrix((data2, col_pos, row_pos_ind))

print('COO 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인')
print(sparse_coo.toarray())
print('CSR 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인')
print(sparse_csr.toarray())

COO 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인
[[0 0 1 0 0 5]
 [1 4 0 3 2 5]
 [0 6 0 3 0 0]
 [2 0 0 0 0 0]
 [0 0 0 7 0 8]
 [1 0 0 0 0 0]]
CSR 변환된 데이터가 제대로 되었는지 다시 Dense로 출력 확인
[[0 0 1 0 0 5]
 [1 4 0 3 2 5]
 [0 6 0 3 0 0]
 [2 0 0 0 0 0]
 [0 0 0 7 0 8]
 [1 0 0 0 0 0]]


실사용 시에는 밀집 행렬을 생성 파라미터로 입력 시 희소 행렬로 생성함.

In [17]:
dense3 = np.array([[0,0,1,0,0,5], [1,4,0,3,2,5], [0,6,0,3,0,0], [2,0,0,0,0,0], [0,0,0,7,0,8], [1,0,0,0,0,0]])

coo = sparse.coo_matrix(dense3)
csr = sparse.csr_matrix(dense3)